# Détection d'anomalies routières — Étape 5 : Déploiement

**Notebook :** `05_deploiement.ipynb`

Le benchmark (étape 4) a retenu **C. DeepLabV3+ pré-entraîné** : meilleure AP (0,830), plus
stable d'un tour à l'autre, meilleure généralisation aux pneus jamais vus, et le plus léger des
deux modèles supervisés. Cette étape le transforme en **application utilisable** :

| § | Étape | Résultat |
|---|---|---|
| 0 | Chargement | données + `modele_retenu.json` de l'étape 4 |
| 1 | Entraînement final sur les 60 images | le modèle de production |
| 2 | Export : PyTorch et **ONNX** | un fichier de modèle portable, sans PyTorch |
| 3 | Vérification de l'export et mesure de vitesse | ONNX = PyTorch ; temps d'inférence |
| 4 | **Application web Flask** : test de l'API et ouverture dans Colab | l'application du projet, avec ce modèle |
| 5 | Paquet du modèle | `model.onnx` + `config.json`, à placer dans l'application sur votre PC |

**GPU recommandé** pour l'entraînement final (≈ 3 minutes sur T4).

## 0. Chargement

Sélectionnez **en une seule fois** : `RoadAnomaly.zip`, `roadanomaly_lib.py` et
`resultats_etape4.zip` (qui contient `modele_retenu.json`).

In [ ]:
import os, sys, zipfile, shutil, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BASE = Path("/content") if IN_COLAB else Path.cwd()
OUT = BASE / "resultats_etape5"; OUT.mkdir(exist_ok=True)

def trouver_dataset(dossier):
    dossier = Path(dossier)
    hits = sorted(dossier.rglob("frame_list.json")) if dossier.exists() else []
    return hits[0].parent if hits else None

if IN_COLAB:
    os.chdir(BASE)
    if trouver_dataset(BASE / "data") is None or not (BASE / "etape4" / "modele_retenu.json").exists():
        from google.colab import files
        print("Sélectionnez RoadAnomaly.zip, roadanomaly_lib.py et resultats_etape4.zip :")
        for nom in files.upload():
            p = BASE / nom
            if nom.startswith("RoadAnomaly") and nom.endswith(".zip"):
                with zipfile.ZipFile(p) as z: z.extractall(BASE / "data")
                os.remove(p)
            elif nom.startswith("resultats_etape4") and nom.endswith(".zip"):
                with zipfile.ZipFile(p) as z: z.extractall(BASE / "etape4")
                os.remove(p)
            elif nom.startswith("roadanomaly_lib") and nom.endswith(".py") and nom != "roadanomaly_lib.py":
                shutil.move(p, BASE / "roadanomaly_lib.py")
    DATA_ROOT = trouver_dataset(BASE / "data")
else:
    DATA_ROOT = next((p for p in [Path("RoadAnomaly"), Path("../RoadAnomaly")]
                      if (p / "frame_list.json").exists()), None)

assert DATA_ROOT is not None, "Dataset introuvable : envoyez RoadAnomaly.zip"
assert (BASE / "roadanomaly_lib.py").exists(), "roadanomaly_lib.py manquant"
assert (BASE / "etape4" / "modele_retenu.json").exists(), "modele_retenu.json manquant : envoyez resultats_etape4.zip"
BEST = json.load(open(BASE / "etape4" / "modele_retenu.json", encoding="utf-8"))
print(json.dumps(BEST, indent=1, ensure_ascii=False))

In [ ]:
import importlib
def ensure(module, pip_name):
    try:
        importlib.import_module(module)
    except ImportError:
        %pip install -q {pip_name}
        importlib.invalidate_caches()
ensure("segmentation_models_pytorch", "segmentation-models-pytorch==0.6.0")
ensure("onnxruntime", "onnxruntime")
ensure("onnx", "onnx")
ensure("flask", "flask")
ensure("markdown", "markdown")
try:
    import albumentations
    assert int(albumentations.__version__.split(".")[0]) >= 2
except (ImportError, AssertionError):
    %pip install -q "albumentations==2.0.8"
    importlib.invalidate_caches()

import time, numpy as np, pandas as pd, torch, torch.nn as nn, cv2, matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
sys.path.insert(0, str(BASE))
import roadanomaly_lib; importlib.reload(roadanomaly_lib)
from roadanomaly_lib import *
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
frames = list_frames(DATA_ROOT)
print("appareil :", device, "| images :", len(frames))
plt.rcParams.update({"figure.dpi": 80, "savefig.dpi": 150, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "axes.axisbelow": True})

## 1. Entraînement final sur les 60 images

### Pourquoi réentraîner ?

Pendant la validation croisée, chaque modèle n'a vu que ≈ 36 images : 24 étaient réservées à la
validation et au test. Pour le modèle de production, on utilise **toutes les données** : plus
d'exemples, donc un meilleur modèle. C'est la pratique standard.

### Sans validation, comment choisir l'époque et le seuil ?

Il n'y a plus d'images de côté. On reprend donc les réglages **mesurés pendant la validation
croisée** (`modele_retenu.json`) :

- **nombre d'époques** = la meilleure époque moyenne des 5 tours. On garde le même planning
  (décroissance cosinus prévue sur 40 époques), arrêté à cette époque, pour reproduire les
  conditions qui ont donné les meilleurs modèles ;
- **seuil de décision** = le seuil moyen choisi sur les validations des 5 tours.

### Quelle performance annoncer ?

Le modèle final n'a **plus de jeu de test** : toutes les images ont servi à l'entraîner.
La performance à annoncer est donc **celle de la validation croisée** (AP 0,830, IoU
0,656 ± 0,055), qui estime ce que ce type de modèle obtient sur des images qu'il n'a pas vues.
Évaluer le modèle final sur ses propres images d'entraînement donnerait un score trompeur.

In [ ]:
assert BEST["architecture"] in ("Unet", "DeepLabV3Plus"), \
    "Ce notebook déploie un modèle supervisé (A, B ou C) ; adaptez-le pour un autre choix."
FINAL_EPOCHS = max(1, int(round(BEST["meilleure_epoque_moyenne"])))
THRESHOLD = float(BEST["seuil_moyen"])
PRETRAINED = BEST["key"] != "A_unet_scratch"
ENCODER_WEIGHTS = None if (os.environ.get("RA_NO_PRETRAINED") or not PRETRAINED) else "imagenet"

class FinalConfig:
    epochs_schedule = 40                  # même planning cosinus qu'en validation croisée
    epochs = FINAL_EPOCHS                 # arrêt à la meilleure époque moyenne
    freeze_epochs = 3
    batch_size = 4
    samples_per_image = 4
    lr_decoder = 1e-3
    lr_encoder = 1e-4 if PRETRAINED else 1e-3

print(f"Modèle : {BEST['label']} | {FinalConfig.epochs} époques | seuil {THRESHOLD:.3f}")

In [ ]:
def build_model():
    Arch = getattr(smp, BEST["architecture"])
    return Arch(encoder_name=BEST["encodeur"], encoder_weights=ENCODER_WEIGHTS, in_channels=3, classes=1)

dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)
bce_loss = nn.BCEWithLogitsLoss()
criterion = lambda logits, y: bce_loss(logits, y) + dice_loss(logits, y)

import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
cfg = FinalConfig
ds = RoadAnomalyDataset(DATA_ROOT, frames, get_train_transform(), cfg.samples_per_image)
loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, drop_last=True)
model = build_model().to(device)
enc = list(model.encoder.parameters())
dec = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
opt = torch.optim.AdamW([{"params": enc, "lr": cfg.lr_encoder}, {"params": dec, "lr": cfg.lr_decoder}],
                        weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs_schedule)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

hist, t0 = [], time.time()
for epoch in range(1, cfg.epochs + 1):
    frozen = PRETRAINED and epoch <= cfg.freeze_epochs
    for p in enc:
        p.requires_grad = not frozen
    model.train()
    if frozen:
        model.encoder.eval()
    tot, n = 0.0, 0
    for b in loader:
        x, y = b["image"].to(device), b["mask"].to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item() * len(x); n += len(x)
    sched.step()
    hist.append(tot / n)
    print(f"époque {epoch:2d}/{cfg.epochs}  perte {tot / n:.3f}{'  (encodeur gelé)' if frozen else ''}")
print(f"Entraînement final : {(time.time() - t0) / 60:.1f} min sur {len(frames)} images")
model.eval()

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.plot(range(1, len(hist) + 1), hist, color="#2a78d6", lw=2)
ax.set_xlabel("Époque"); ax.set_ylabel("Perte (BCE + Dice)"); ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.set_title("Entraînement final sur les 60 images", fontweight="bold")
plt.tight_layout(); fig.savefig(OUT / "courbe_entrainement_final.png"); plt.show()

## 2. Export du modèle

Deux formats :

| Fichier | Format | Usage |
|---|---|---|
| `model.pt` | poids PyTorch (`state_dict`) | réentraîner ou modifier le modèle plus tard |
| `model.onnx` | **ONNX** (*Open Neural Network Exchange*) | le **déploiement** : le modèle s'exécute avec `onnxruntime`, une bibliothèque légère, **sans PyTorch**, sur CPU ou GPU, et même depuis d'autres langages (C++, C#, JavaScript…) |

L'export ONNX déclare la hauteur et la largeur de l'image comme **dimensions variables** :
l'application accepte des images de toute taille.

`config.json` rassemble tout ce dont l'application a besoin : architecture, seuil, normalisation,
résolution de travail et performances de validation croisée (affichées dans l'application).

In [ ]:
APP_DIR = OUT / "modele_objectif_1"; APP_DIR.mkdir(exist_ok=True)
torch.save(model.state_dict(), OUT / "model.pt")

model_cpu = build_model().eval()
model_cpu.load_state_dict({k: v.cpu() for k, v in model.state_dict().items()})
dummy = torch.zeros(1, 3, 544, 960)
export_args = dict(input_names=["image"], output_names=["logits"], opset_version=17,
                   dynamic_axes={"image": {2: "hauteur", 3: "largeur"}, "logits": {2: "hauteur", 3: "largeur"}})
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    try:
        torch.onnx.export(model_cpu, dummy, str(APP_DIR / "model.onnx"), dynamo=False, **export_args)
    except TypeError:                                # anciennes versions de PyTorch : pas d'argument dynamo
        torch.onnx.export(model_cpu, dummy, str(APP_DIR / "model.onnx"), **export_args)

config = {"label": BEST["label"], "architecture": BEST["architecture"], "encoder": BEST["encodeur"],
          "threshold": THRESHOLD, "work_width": WORK_W, "mean": list(MEAN), "std": list(STD),
          "cv_AP": BEST["AP"], "cv_IoU_mean": BEST["IoU_tours_moyenne"], "cv_IoU_std": BEST["IoU_tours_ecart_type"],
          "final_epochs": FINAL_EPOCHS, "trained_on_images": len(frames),
          "dataset": "RoadAnomaly (Lis et al., ICCV 2019)"}
json.dump(config, open(APP_DIR / "config.json", "w", encoding="utf-8"), indent=1, ensure_ascii=False)
for f in ["model.onnx", "config.json"]:
    print(f"{f:<12} {(APP_DIR / f).stat().st_size / 1e6:6.1f} Mo")
print(f"{'model.pt':<12} {(OUT / 'model.pt').stat().st_size / 1e6:6.1f} Mo")

## 3. Vérification de l'export et vitesse

Un export peut silencieusement changer les résultats. On compare donc les sorties ONNX et
PyTorch sur de vraies images, à deux tailles, puis on mesure le temps d'inférence.

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession(str(APP_DIR / "model.onnx"), providers=["CPUExecutionProvider"])
norm = lambda im: ((im.astype(np.float32) / 255 - np.array(MEAN, np.float32)) / np.array(STD, np.float32)).transpose(2, 0, 1)[None]

rows = []
for n in frames[:3]:
    img = load_image(DATA_ROOT, n)
    for (W_, H_) in [(960, 544), (1280, 736)]:
        x = norm(cv2.resize(img, (W_, H_), interpolation=cv2.INTER_AREA))
        o = sess.run(None, {"image": x})[0]
        with torch.no_grad():
            t = model_cpu(torch.from_numpy(x)).numpy()
        rows.append({"image": n[:25], "taille": f"{W_}x{H_}", "écart max (logits)": float(np.abs(o - t).max()),
                     "décisions identiques": float(((o > 0) == (t > 0)).mean())})
check = pd.DataFrame(rows); display(check)
assert check["décisions identiques"].min() > 0.999, "l'export ONNX ne reproduit pas le modèle PyTorch"
print("Export ONNX vérifié : mêmes décisions que PyTorch.")

x = norm(cv2.resize(load_image(DATA_ROOT, frames[0]), (960, 544)))
def chrono(fn, rep=5):
    fn(); t0 = time.time()
    for _ in range(rep): fn()
    return (time.time() - t0) / rep * 1000
vit = {"ONNX, CPU": chrono(lambda: sess.run(None, {"image": x}))}
if device.type == "cuda":
    xg = torch.from_numpy(x).to(device)
    def gpu():
        with torch.no_grad(): model(xg); torch.cuda.synchronize()
    vit["PyTorch, GPU"] = chrono(gpu, 20)
for k, v in vit.items():
    print(f"Temps par image 960x544 — {k:<13}: {v:6.0f} ms  ({1000 / v:5.1f} images/s)")
json.dump(vit, open(OUT / "vitesse_inference.json", "w"), indent=1, ensure_ascii=False)

## 4. Application web (Flask)

### Pourquoi Flask ?

**Flask** est un petit *framework* web en Python : il fait tourner un **serveur** qui répond aux
requêtes du navigateur. Contrairement à un outil d'interface tout fait, on écrit soi-même les pages
(HTML, CSS, JavaScript). C'est plus de travail, mais on contrôle entièrement l'interface, et l'application
regroupe **les 6 objectifs du groupe** dans une seule page, avec une barre de navigation.

### Comment l'application est organisée

L'application est dans le dossier `application/` du dépôt GitHub du projet :

| Élément | Rôle |
|---|---|
| `app.py` | le **serveur Flask** : charge chaque modèle une seule fois au démarrage, puis répond aux requêtes |
| `objectifs/objectif_1.py` | **notre objectif** : prétraitement, modèle ONNX, seuil, zones, dessin |
| `objectifs/objectif_2.py` … `objectif_6.py` | les objectifs des autres membres |
| `templates/` | les pages HTML (bannière, barre des 6 objectifs, page de chaque objectif) |
| `static/` | le thème (`style.css`) et le JavaScript de la page |
| `modeles/objectif_1/` | **`model.onnx` et `config.json` produits par ce notebook** |

### Ce qui se passe quand on analyse une photo

1. Le navigateur envoie la photo au serveur : requête **POST** vers `/api/objectif/1/analyser`.
2. Le serveur la ramène à 960 px de large (la résolution d'entraînement), la normalise et la passe
   au **modèle ONNX** (avec `onnxruntime`, sans PyTorch).
3. Il remet la probabilité à la taille d'origine, applique le **seuil**, sépare les **zones**
   (composantes connexes) et mesure pour chacune sa position, sa surface et la confiance du modèle.
4. Il répond en **JSON** : les images (résultat et carte de probabilité, encodées en base64), les
   chiffres clés et la liste des zones.
5. Le JavaScript met la page à jour sans la recharger. Quand on change le seuil, le serveur
   réutilise la probabilité déjà calculée : le modèle n'est pas relancé.

### Récupérer l'application

La cellule suivante télécharge l'application depuis le dépôt GitHub du groupe et y place le modèle
qu'on vient d'exporter. Si le dépôt n'est pas accessible, elle demande d'envoyer
`app_roadanomaly_flask.zip`.

In [ ]:
import subprocess
DEPOT = "https://github.com/RouaLz/projet-deeplearning.git"
SRC_APP = BASE / "projet-deeplearning"
if not SRC_APP.exists():
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", DEPOT, str(SRC_APP)], capture_output=True, text=True)
    if r.returncode != 0:
        print("Dépôt inaccessible :", r.stderr.strip()[-300:])

# le dossier de l'application : celui qui contient app.py et objectifs/
candidats = [p.parent for p in sorted(BASE.rglob("app.py")) if (p.parent / "objectifs" / "objectif_1.py").exists()]
if not candidats and IN_COLAB:
    from google.colab import files
    print("Sélectionnez app_roadanomaly_flask.zip")
    for nom in files.upload():
        with zipfile.ZipFile(nom) as z:
            z.extractall(BASE / "app_flask")
    candidats = [p.parent for p in sorted(BASE.rglob("app.py")) if (p.parent / "objectifs" / "objectif_1.py").exists()]
assert candidats, "Application Flask introuvable"
FLASK_DIR = candidats[0]

# le modèle exporté section 2 va dans modeles/objectif_1/
dest = FLASK_DIR / "modeles" / "objectif_1"; dest.mkdir(parents=True, exist_ok=True)
for f in ["model.onnx", "config.json"]:
    shutil.copy(APP_DIR / f, dest / f)
print("Application :", FLASK_DIR)
for d in sorted(FLASK_DIR.iterdir()):
    if not d.name.startswith((".", "__")):
        print("  ", d.name + ("/" if d.is_dir() else ""))

### Tester l'API sans navigateur

Flask fournit un **client de test** : il envoie des requêtes au serveur directement depuis Python,
sans démarrer de site web. On envoie une image du dataset, exactement comme le ferait le navigateur,
et on lit la réponse JSON.

In [ ]:
import base64, io
sys.path.insert(0, str(FLASK_DIR)); os.chdir(FLASK_DIR)
import app as appli                                   # démarre le serveur : charge les modèles
os.chdir(BASE)
client = appli.app.test_client()

page = client.get("/objectif/1")
print("Page de l'objectif 1 :", page.status_code, "(200 = OK)")

def b64_vers_image(texte):
    return plt.imread(io.BytesIO(base64.b64decode(texte.split(",", 1)[1])), format="jpeg")

for nom_image in ["animals22_red_fox_on_road", "animals05_cows_near_Phonsavan_Laos"]:
    if nom_image not in frames:
        continue
    octets = cv2.imencode(".jpg", load_image(DATA_ROOT, nom_image)[..., ::-1])[1].tobytes()
    rep = client.post("/api/objectif/1/analyser",
                      data={"image": (io.BytesIO(octets), "photo.jpg"), "seuil": str(THRESHOLD)},
                      content_type="multipart/form-data")
    res = rep.get_json()
    print(f"\n{nom_image} — statut {rep.status_code} — {res['nb_zones']} zone(s), "
          f"{res['surface']} de l'image, analyse en {res['ms']} ms")
    for z in res["zones"][:5]:
        print(f"   zone {z['numero']} : {z['position']:<22} surface {z['surface']:>8}   confiance {z['confiance']:>5} ({z['niveau']})")
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
    for a, im, t in zip(axes, [b64_vers_image(res["resultat"]), b64_vers_image(res["carte"])],
                        ["Réponse de l'API : anomalies détectées", "Carte de probabilité"]):
        a.imshow(im); a.set_title(t); a.axis("off")
    plt.tight_layout(); fig.savefig(OUT / f"exemple_application_{nom_image}.png"); plt.show()

### Ouvrir l'application dans Colab

La cellule suivante démarre le serveur Flask en arrière-plan et l'ouvre dans une **fenêtre
Colab** : c'est l'application complète (barre des 6 objectifs, exemples, zones cliquables).
Testez des **photos que le modèle n'a jamais vues** (cherchez par exemple « cow on road » ou
« rock on highway » sur un moteur de recherche d'images) : les images du dataset ont servi à
l'entraîner, elles donneraient une impression trop optimiste.

Ce lien ne fonctionne que **pour vous** et **tant que la session Colab tourne**. Pour la
soutenance, lancez l'application sur votre ordinateur (section 5).

In [ ]:
import threading
if IN_COLAB:
    threading.Thread(target=lambda: appli.app.run(host="127.0.0.1", port=5000, use_reloader=False),
                     daemon=True).start()
    time.sleep(2)
    from google.colab import output
    output.serve_kernel_port_as_window(5000, anchor_text="Ouvrir l'application (cliquez ici)")
else:
    print("Hors Colab : lancez « python app.py » dans le dossier", FLASK_DIR, "puis ouvrez http://127.0.0.1:5000")

## 5. Paquet du modèle et exécution sur votre ordinateur

L'application Flask (le code) est dans le dépôt GitHub du projet. Ce qui sort de ce notebook,
c'est **le modèle** : deux fichiers, rassemblés dans `modele_objectif_1.zip`.

| Fichier | Rôle |
|---|---|
| `model.onnx` | le modèle (≈ 90 Mo) |
| `config.json` | seuil, normalisation, résolution de travail, performances |

Ils ne sont **pas dans le dépôt Git** (GitHub refuse les fichiers de plus de 100 Mo, et un modèle
n'a rien à faire dans l'historique du code) : ils sont publiés dans la **Release** du dépôt.

### Lancer l'application sur votre PC (gratuit, sans internet)

1. Récupérez le projet : `git clone https://github.com/RouaLz/projet-deeplearning.git`
2. Décompressez `modele_objectif_1.zip` et copiez `model.onnx` et `config.json` dans
   `application/modeles/objectif_1/`.
3. Ouvrez un terminal dans le dossier `application`, puis :
   ```
   pip install -r requirements.txt
   python app.py
   ```
4. Ouvrez **http://127.0.0.1:5000** dans votre navigateur. Pour arrêter : **Ctrl + C**.

C'est la solution la plus sûre pour une **démonstration en soutenance** : elle ne dépend ni
d'internet ni d'une session Colab.

In [ ]:
# Téléchargements : 1) le modèle pour l'application (≈ 85 Mo) ; 2) les poids PyTorch (≈ 90 Mo) ;
# 3) les figures et mesures de l'étape 5 (léger)
shutil.make_archive(str(OUT / "modele_objectif_1"), "zip", APP_DIR)
LEGER = BASE / "resultats_etape5_figures"; LEGER.mkdir(exist_ok=True)
for f in OUT.iterdir():
    if f.suffix in (".png", ".json"):
        shutil.copy(f, LEGER / f.name)
shutil.copy(APP_DIR / "config.json", LEGER / "config.json")
shutil.make_archive(str(BASE / "resultats_etape5_figures"), "zip", LEGER)
for f in [OUT / "modele_objectif_1.zip", OUT / "model.pt", BASE / "resultats_etape5_figures.zip"]:
    print(f"{f.name:<30} {f.stat().st_size / 1e6:7.1f} Mo")
if IN_COLAB:
    from google.colab import files
    files.download(str(OUT / "modele_objectif_1.zip"))
    files.download(str(OUT / "model.pt"))
    files.download(str(BASE / "resultats_etape5_figures.zip"))

## Synthèse du déploiement

| Élément | Choix | Justification |
|---|---|---|
| Modèle | DeepLabV3+ / ResNet34, réentraîné sur les 60 images | retenu au benchmark ; plus de données pour le modèle final |
| Époques, seuil | moyennes de la validation croisée | plus de jeu de validation disponible |
| Performance annoncée | celle de la validation croisée | le modèle final n'a plus de jeu de test |
| Format | ONNX + onnxruntime | portable, léger, sans PyTorch, sorties vérifiées identiques |
| Interface | application web **Flask** : image, seuil réglable, zones numérotées et cliquables, carte de probabilité | contrôle complet de l'interface ; une seule application pour les 6 objectifs du groupe |
| Architecture | serveur Flask + API JSON (`/api/objectif/N/analyser`) + page HTML/JavaScript | le modèle est chargé une fois ; l'API peut servir à d'autres clients (application mobile, script) |
| Diffusion | exécution locale (`python app.py`), code sur GitHub, modèle dans la Release | gratuit ; démonstration fiable sans internet ; reproductible par le groupe |

**Pour le rapport (limites et pistes) :**

- **Vitesse :** sur un processeur (CPU), une image prend de l'ordre de la seconde (voir la
  mesure section 3) : très bien pour une démonstration, insuffisant pour un véhicule, qui
  demanderait un GPU embarqué et une version allégée (encodeur plus petit, quantification INT8,
  TensorRT).
- **Domaine :** 60 images, de jour, par beau temps pour la plupart. Les performances sur la nuit,
  la pluie ou d'autres caméras sont inconnues.
- **Suivi :** en production réelle, il faudrait collecter les erreurs signalées par les
  utilisateurs pour enrichir le dataset et réentraîner.